# 03 — Modulation by Fifth

**Hypothesis:** The C→G modulation swaps F (e4, positive) ↔ F♯ (eA, negative). The operator performing this swap is a hyperbolic rotor `exp(t · e4 ∧ eA)` for some `t`, and it maps the C-diatonic 7-blade to the G-diatonic 7-blade (up to sign/orientation).

In [1]:
from kingdon import Algebra
import numpy as np

alg = Algebra(7, 5, 1)
def mv(k): return alg.multivector({k: 1})

PITCH = {
    'C':  mv('e1'), 'D':  mv('e2'), 'E':  mv('e3'), 'F':  mv('e4'),
    'G':  mv('e5'), 'A':  mv('e6'), 'B':  mv('e7'),
    'Cs': mv('e8'), 'Ds': mv('e9'), 'Fs': mv('eA'), 'Gs': mv('eB'), 'As': mv('eC'),
}
OCTAVE = mv('e0')

def norm2(x): return (x * ~x).e
def is_zero(x, tol=1e-10): return all(abs(float(v)) <= tol for v in x.values())
def blade_coeffs(x, tol=1e-10): return {alg.bin2canon[k]: float(v) for k, v in x.items() if abs(float(v)) > tol}
def sandwich(R, x): return R * x * ~R

print('Setup ready.')

Setup ready.


In [2]:
# The bivector for the F <-> Fs plane
# e4 (F, +1) and eA (Fs, -1) span a split-signature 2D subspace
# (e4·eA)² = e4·eA·e4·eA = -(e4²)·(eA²) = -(+1)(-1) = +1  → hyperbolic
B = PITCH['F'] * PITCH['Fs']
B_sq = (B * B).e
print(f'B = e4*eA,  B² = {B_sq}')
print('✓ hyperbolic bivector (B² = +1)' if B_sq == 1 else f'✗ B² = {B_sq}')

B = e4*eA,  B² = 1
✓ hyperbolic bivector (B² = +1)


In [3]:
# Hyperbolic rotor: exp(t·B) = cosh(t) + sinh(t)·B
# Sandwich R·e4·~R: with R = cosh(t) + sinh(t)·B
# Maps e4 → cosh(2t)·e4 + sinh(2t)·(−eA) in the split plane
# (exact formula depends on commutation; check numerically)

scalar1 = alg.multivector({'e': 1})

print(f'  {"t":>6}  {"e4(F) coeff":>14}  {"eA(Fs) coeff":>14}')
print('  ' + '-'*40)
for t in [0.0, 0.5, 1.0, np.pi/4, np.pi/2, 2.0, 3.0]:
    R = np.cosh(t) * scalar1 + np.sinh(t) * B
    img_F = sandwich(R, PITCH['F'])
    bc = blade_coeffs(img_F)
    c_e4 = bc.get('e4', 0.0)
    c_eA = bc.get('eA', 0.0)
    print(f'  {t:>6.3f}  {c_e4:>14.6f}  {c_eA:>14.6f}')

       t     e4(F) coeff    eA(Fs) coeff
  ----------------------------------------
   0.000        1.000000        0.000000
   0.500        1.543081       -1.175201
   1.000        3.762196       -3.626860
   0.785        2.509178       -2.301299
   1.571       11.591953      -11.548739
   2.000       27.308233      -27.289917
   3.000      201.715636     -201.713157


In [4]:
# Claim: pure swap (F→Fs, Fs→F) is unreachable by real hyperbolic rotor
# Requires cosh(2t) = 0 → impossible for real t
print('A pure swap requires cosh(2t) = 0 — no real solution.')
print('✓ confirmed: hyperbolic rotor interpolates continuously, never achieves pure swap.')

A pure swap requires cosh(2t) = 0 — no real solution.
✓ confirmed: hyperbolic rotor interpolates continuously, never achieves pure swap.


In [5]:
# Build C-diatonic and G-diatonic 7-blades
C_diatonic = (PITCH['C'] ^ PITCH['D'] ^ PITCH['E'] ^ PITCH['F'] ^
              PITCH['G'] ^ PITCH['A'] ^ PITCH['B'])
G_diatonic = (PITCH['C'] ^ PITCH['D'] ^ PITCH['E'] ^ PITCH['Fs'] ^
              PITCH['G'] ^ PITCH['A'] ^ PITCH['B'])

print('C-diatonic:', blade_coeffs(C_diatonic))
print('G-diatonic:', blade_coeffs(G_diatonic))

C-diatonic: {'e1234567': 1.0}
G-diatonic: {'e123567A': -1.0}


In [6]:
# Apply sandwich to C_diatonic: does the eA(Fs) component grow?
scalar1 = alg.multivector({'e': 1})

t_test = np.pi / 4
R_test = np.cosh(t_test) * scalar1 + np.sinh(t_test) * B

img = sandwich(R_test, C_diatonic)
img_bc = blade_coeffs(img)

print(f'Sandwich of C_diatonic at t = π/4:')
for k, v in sorted(img_bc.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {k}: {v:.6f}')

print()
# Check rotor is unit
R_norm = (R_test * ~R_test)
print(f'R·~R = {blade_coeffs(R_norm)}')
print('✓ unit rotor' if abs(float(R_norm.e) - 1.0) < 1e-10 and len(blade_coeffs(R_norm)) == 1 else '✗')

Sandwich of C_diatonic at t = π/4:
  e1234567: 2.509178
  e123567A: 2.301299

R·~R = {'e': 0.9999999999999999}
✓ unit rotor


In [7]:
# Verify: at large t the C_diatonic blade approaches G_diatonic in the F/Fs components
C_key = list(blade_coeffs(C_diatonic).keys())[0]
G_key = list(blade_coeffs(G_diatonic).keys())[0]

print(f'C-diatonic blade key: {C_key}')
print(f'G-diatonic blade key: {G_key}')
print()
print(f'  {"t":>6}  {"C-blade coeff":>16}  {"G-blade coeff":>16}')
print('  ' + '-'*42)
for t in [0.0, 0.5, 1.0, 2.0, 4.0, 6.0]:
    R = np.cosh(t) * scalar1 + np.sinh(t) * B
    img = sandwich(R, C_diatonic)
    bc = blade_coeffs(img, tol=0)
    c_c = bc.get(C_key, 0.0)
    c_g = bc.get(G_key, 0.0)
    print(f'  {t:>6.2f}  {c_c:>16.6f}  {c_g:>16.6f}')

C-diatonic blade key: e1234567
G-diatonic blade key: e123567A

       t     C-blade coeff     G-blade coeff
  ------------------------------------------
    0.00          1.000000          0.000000
    0.50          1.543081          1.175201
    1.00          3.762196          3.626860
    2.00         27.308233         27.289917
    4.00       1490.479161       1490.478826
    6.00      81377.395713      81377.395706


## Discussion

The bivector `e4 ∧ eA` (F ∧ F♯) squares to +1, confirming a **hyperbolic** (boost-like) generator. The rotor `exp(t · e4·eA)` moves the C-diatonic blade continuously toward the G-diatonic blade: the F-component decays as cosh(2t)⁻¹ and the F♯-component grows, but a pure swap is unreachable for finite real t.

**Partial hold:** The rotor does interpolate between the two diatonic frames, and the direction of motion is exactly right (F → F♯). But modulation is a continuous deformation in this algebra, not a discrete operation. C and G are separated by an infinite boost parameter — they live on opposite sides of a hyperbolic sheet.

This is informative: the algebra encodes modulation as a **metric-altering, boost-like** operation, not an isometry. The C and G diatonic frames are genuinely in different metric regions.